# Answer-only precommit v2.2 — code-only A100 launcher
Select an **A100** runtime. This notebook clones public code and generates your own private outputs. It contains no historical results. Use a fresh runtime/output directory; the runners refuse to overwrite records.

Set `REVISION` to an exact Git commit to freeze an experiment. `main` is convenient for exploration; the actual checkout and source hashes are recorded. Seed is **123**. The bounded suite runs iLLaDA then Dream, up to 3600 seconds each. Mediation, subspace patching and B-first persistence are not implemented by v2.2.


In [ ]:
from pathlib import Path
import subprocess, sys, json
REVISION = "main"  # Replace with an exact commit for a frozen run.
ROOT = Path("/content/DLM_Multitask_Reasoning")
if ROOT.exists():
    raise FileExistsError("Use a fresh runtime or preserve the existing checkout before a new run.")
subprocess.run(["git", "clone", "https://github.com/Twist-Shan/DLM_Multitask_Reasoning.git", str(ROOT)], check=True)
subprocess.run(["git", "checkout", "--detach", REVISION], cwd=ROOT, check=True)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
print("Checked out", COMMIT)


## Environment and mechanical checks
The isolated environment inherits Colab CUDA PyTorch while pinning the model-interface dependencies. No model weights are loaded by these tests.


In [ ]:
subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(ROOT / ".colab-env")], check=True)
PY = str(ROOT / ".colab-env/bin/python")
subprocess.run([PY, "-m", "pip", "install", "-e", str(ROOT), "-r", str(ROOT / "requirements/precommit.txt")], check=True)
subprocess.run([PY, "-m", "unittest", "discover", "-s", "tests", "-p", "test_precommit*.py", "-v"], cwd=ROOT, check=True)
subprocess.run([PY, "scripts/package_precommit_code.py", "--out", "/content/precommit-v22-code.zip"], cwd=ROOT, check=True)
RUNS = ROOT / "runs"
RUNS.mkdir(exist_ok=False)
(RUNS / "checkout.json").write_text(json.dumps({"revision_requested": REVISION, "commit": COMMIT}, indent=2))
(RUNS / "environment.freeze.txt").write_bytes(subprocess.check_output([PY, "-m", "pip", "freeze"]))


## Run the bounded A100 suite
This cell starts paid GPU computation. It checks that the device is an A100, preserves partial results after failures and runs both models sequentially. Logs and the archive stay in your private Colab runtime.


In [ ]:
subprocess.run([PY, "scripts/colab_precommit_v22_suite.py"], cwd=ROOT, check=True, timeout=7500)
status = json.loads((RUNS / "suite_status.json").read_text())
print(json.dumps(status, indent=2))
if status["status"] != "finished":
    raise RuntimeError("Suite incomplete. Preserve and download the partial archive; inspect model logs.")


## Audit saved probabilities
Run after a complete suite. All diagnostic outputs are local to this runtime and contain your own results.


In [ ]:
for script, name in [("analyze_precommit_v22.py", "analysis.json"),
                     ("audit_precommit_v22_candidates.py", "candidates.json"),
                     ("audit_precommit_v22_carry.py", "carry.json")]:
    subprocess.run([PY, "scripts/" + script, str(RUNS), "--out", str(RUNS / name)], cwd=ROOT, check=True)


## Save outputs, then release the GPU
The final ZIP includes code provenance, raw outputs and your audit files. Download and verify it locally before disconnecting and deleting the runtime. No result is pushed to GitHub. This download cell also works after a failed run if the earlier setup finished.


In [ ]:
import hashlib, shutil
from google.colab import files
archive = Path(shutil.make_archive("/content/precommit-v22-with-audits", "zip", ROOT, "runs"))
print("bytes", archive.stat().st_size, "sha256", hashlib.sha256(archive.read_bytes()).hexdigest())
files.download(str(archive))
